# 書 KiriCallig — Кириллическая каллиграфия

**Генерация кириллического шрифта в стиле японской каллиграфии** с помощью [VecGlypher](https://github.com/xk-huang/VecGlypher) — LLM для генерации векторных глифов.

## Что делает этот ноутбук

1. Устанавливает VecGlypher и все зависимости
2. Скачивает модель VecGlypher-27b-it с HuggingFace
3. Запускает vLLM-сервер для инференса
4. Генерирует SVG-глифы для всех кириллических символов
5. Собирает .ttf шрифт из SVG
6. Скачивает готовый шрифт

**GPU:** T4 (бесплатный Colab, с 4-bit квантизацией) или A100/L4 (Colab Pro, полная точность)

---

## 0. Настройки

Выбери стиль каллиграфии и параметры генерации:

In [ ]:
#@title Параметры генерации { display-mode: "form" }

#@markdown ### Стиль каллиграфии
STYLE = "default"  #@param ["default", "kaisho", "gyosho", "sosho", "modern"]
#@markdown - **default** — сбалансированный шодо, динамичные штрихи
#@markdown - **kaisho** (楷書) — формальный, чёткие штрихи
#@markdown - **gyosho** (行書) — полукурсив, плавный
#@markdown - **sosho** (草書) — травяное письмо, экспрессивный
#@markdown - **modern** — минималистичный современный

#@markdown ### Параметры генерации
TEMPERATURE = 0.7  #@param {type: "slider", min: 0.1, max: 1.5, step: 0.1}
TOP_P = 0.8  #@param {type: "slider", min: 0.1, max: 1.0, step: 0.05}
MAX_TOKENS = 1024  #@param {type: "integer"}

#@markdown ### Набор символов
GENERATE_UPPER = True  #@param {type: "boolean"}
GENERATE_LOWER = True  #@param {type: "boolean"}
GENERATE_DIGITS = True  #@param {type: "boolean"}
GENERATE_PUNCTUATION = True  #@param {type: "boolean"}

print(f"Стиль: {STYLE}")
print(f"Температура: {TEMPERATURE}, top_p: {TOP_P}")

## 1. Установка зависимостей

In [ ]:
%%time
# Определяем GPU
import subprocess
gpu_info = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']).decode()
print(f"GPU: {gpu_info.strip()}")

vram_mb = int(subprocess.check_output(
    ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits']
).decode().strip())

USE_QUANTIZATION = vram_mb < 24000  # T4=16GB, нужна квантизация
print(f"VRAM: {vram_mb} MB")
print(f"Квантизация: {'4-bit (bitsandbytes)' if USE_QUANTIZATION else 'нет (достаточно VRAM)'}")

In [ ]:
%%time
# Установка пакетов
!pip install -q vllm==0.8.5.post1 2>&1 | tail -1
!pip install -q transformers==4.57.3 huggingface_hub 2>&1 | tail -1
!pip install -q fonttools svgpathtools 2>&1 | tail -1
!pip install -q bitsandbytes>=0.45.0 2>&1 | tail -1
!pip install -q openai aiohttp 2>&1 | tail -1

print("\nУстановка завершена!")

## 2. Подготовка символов и стилей

In [ ]:
# Кириллические символы
CYRILLIC_UPPER = list("АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯЁ")
CYRILLIC_LOWER = list("абвгдежзийклмнопрстуфхцчшщъыьэюяё")
DIGITS = list("0123456789")
PUNCTUATION = list(".,;:!?-()«»")

# Собираем набор символов
chars = []
if GENERATE_UPPER: chars += CYRILLIC_UPPER
if GENERATE_LOWER: chars += CYRILLIC_LOWER
if GENERATE_DIGITS: chars += DIGITS
if GENERATE_PUNCTUATION: chars += PUNCTUATION

print(f"Всего символов для генерации: {len(chars)}")
print(f"Символы: {''.join(chars)}")

In [ ]:
# Определения стилей каллиграфии
STYLE_PROMPTS = {
    "default": (
        "display, handwritten, calligraphic, brush-stroke, medium-weight, "
        "expressive, organic, fluid strokes with varied thickness, "
        "inspired by Japanese shodo calligraphy, bold brush dynamics, "
        "elegant flowing curves, ink-brush texture feel, "
        "thick-to-thin stroke transitions, dramatic contrast"
    ),
    "kaisho": (
        "display, handwritten, calligraphic, brush-stroke, bold-weight, "
        "structured, angular, strong vertical strokes, "
        "inspired by Japanese kaisho formal calligraphy, "
        "deliberate brush placement, clear stroke endings, "
        "balanced proportions, strong ink presence"
    ),
    "gyosho": (
        "display, handwritten, calligraphic, brush-stroke, medium-weight, "
        "semi-cursive, flowing, connected strokes, "
        "inspired by Japanese gyosho semi-cursive calligraphy, "
        "fluid transitions between strokes, moderate speed impression, "
        "graceful curves, natural ink flow"
    ),
    "sosho": (
        "display, handwritten, calligraphic, brush-stroke, light-weight, "
        "cursive, highly expressive, abstract, fluid continuous strokes, "
        "inspired by Japanese sosho grass script calligraphy, "
        "rapid brush movement, minimal lifting, "
        "wild energy, extreme thick-thin variation"
    ),
    "modern": (
        "display, handwritten, calligraphic, brush-stroke, regular-weight, "
        "modern, minimal, clean brush strokes, "
        "japanese-inspired minimalist calligraphy, "
        "precise yet organic, restrained elegance, "
        "balanced white space, contemporary feel"
    ),
}

SYSTEM_PROMPT = (
    "You are a specialized vector glyph designer creating SVG path elements.\n\n"
    "CRITICAL REQUIREMENTS:\n"
    "- Each glyph must be a complete, self-contained <path> element\n"
    "- Terminate each <path> element with a newline character\n"
    "- Output ONLY valid SVG <path> elements"
)

style_prompt = STYLE_PROMPTS[STYLE]
print(f"Выбранный стиль: {STYLE}")
print(f"Промпт: {style_prompt[:80]}...")

## 3. Загрузка модели и запуск vLLM-сервера

In [ ]:
%%time
import os

MODEL_ID = "VecGlypher/VecGlypher-27b-it"
PORT = 30000

# Формируем команду запуска vLLM
vllm_cmd = (
    f"python -m vllm.entrypoints.openai.api_server "
    f"--model {MODEL_ID} "
    f"--served-model-name vecglypher "
    f"--host 0.0.0.0 "
    f"--port {PORT} "
    f"--tensor-parallel-size 1 "
    f"--trust-remote-code "
    f"--max-model-len 2048 "
    f"--gpu-memory-utilization 0.92 "
    f"--enforce-eager "  # экономит память
)

if USE_QUANTIZATION:
    vllm_cmd += "--quantization bitsandbytes --load-format bitsandbytes "
    print("Используем 4-bit квантизацию для экономии VRAM")

print(f"Команда: {vllm_cmd}")
print(f"\nЗапускаем vLLM-сервер... (загрузка модели ~5-15 мин)")

In [ ]:
import subprocess
import time
import requests

# Запускаем сервер в фоне
log_file = open("/tmp/vllm_server.log", "w")
server_process = subprocess.Popen(
    vllm_cmd.split(),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env={**os.environ, "VLLM_ATTENTION_BACKEND": "FLASHINFER"}
)
print(f"vLLM PID: {server_process.pid}")

# Ждём готовности
print("Ожидание запуска сервера", end="")
for i in range(300):  # до 5 минут
    try:
        r = requests.get(f"http://localhost:{PORT}/v1/models", timeout=2)
        if r.status_code == 200:
            print(f"\n\nСервер готов! ({i} сек)")
            break
    except:
        pass

    # Проверяем, не упал ли процесс
    if server_process.poll() is not None:
        print(f"\n\nОшибка: сервер упал! Код: {server_process.returncode}")
        print("Лог:")
        !tail -50 /tmp/vllm_server.log
        break

    if i % 10 == 0:
        print(".", end="", flush=True)
    time.sleep(1)
else:
    print("\n\nТаймаут! Проверь лог:")
    !tail -30 /tmp/vllm_server.log

In [ ]:
# Проверка: пробный запрос к серверу
import requests
import json

test_payload = {
    "model": "vecglypher",
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Font design requirements: {style_prompt}\nText content: А"}
    ],
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "max_tokens": MAX_TOKENS,
}

r = requests.post(
    f"http://localhost:{PORT}/v1/chat/completions",
    json=test_payload,
    timeout=120
)

if r.status_code == 200:
    result = r.json()
    test_output = result["choices"][0]["message"]["content"]
    print("Тестовый запрос (буква А):")
    print(test_output[:500])
    print(f"\nТокенов: {result['usage']}")
else:
    print(f"Ошибка: {r.status_code}")
    print(r.text)

## 4. Генерация глифов

In [ ]:
import asyncio
import aiohttp
import json
import re
from pathlib import Path
from IPython.display import clear_output

OUTPUT_DIR = Path("/content/output")
SVG_DIR = OUTPUT_DIR / "svgs" / STYLE
SVG_DIR.mkdir(parents=True, exist_ok=True)

API_URL = f"http://localhost:{PORT}/v1/chat/completions"

# Результаты генерации
results = {}
errors = []


async def generate_glyph(session, char, semaphore):
    """Генерация одного глифа через API."""
    async with semaphore:
        payload = {
            "model": "vecglypher",
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Font design requirements: {style_prompt}\nText content: {char}"}
            ],
            "temperature": TEMPERATURE,
            "top_p": TOP_P,
            "max_tokens": MAX_TOKENS,
        }

        for attempt in range(3):
            try:
                async with session.post(API_URL, json=payload, timeout=aiohttp.ClientTimeout(total=180)) as resp:
                    if resp.status == 200:
                        data = await resp.json()
                        content = data["choices"][0]["message"]["content"]
                        return char, content, None
                    else:
                        error_text = await resp.text()
                        if attempt == 2:
                            return char, None, f"HTTP {resp.status}: {error_text[:200]}"
            except Exception as e:
                if attempt == 2:
                    return char, None, str(e)
                await asyncio.sleep(2 ** attempt)

    return char, None, "Все попытки исчерпаны"


async def generate_all(chars):
    """Генерация всех глифов с параллельными запросами."""
    semaphore = asyncio.Semaphore(4)  # макс 4 одновременных запроса
    completed = 0
    total = len(chars)

    async with aiohttp.ClientSession() as session:
        tasks = [generate_glyph(session, c, semaphore) for c in chars]

        for coro in asyncio.as_completed(tasks):
            char, content, error = await coro
            completed += 1

            if error:
                errors.append((char, error))
                print(f"  [{completed}/{total}] '{char}' — ОШИБКА: {error[:80]}")
            else:
                results[char] = content
                print(f"  [{completed}/{total}] '{char}' (U+{ord(char):04X}) — OK")


print(f"Генерация {len(chars)} глифов в стиле '{STYLE}'...")
print(f"Параллельность: 4 запроса, таймаут: 180с\n")

await generate_all(chars)

print(f"\n{'='*50}")
print(f"Успешно: {len(results)}/{len(chars)}")
if errors:
    print(f"Ошибки: {len(errors)}")
    for char, err in errors:
        print(f"  '{char}': {err[:80]}")

## 5. Извлечение SVG-файлов

In [ ]:
import re

SVG_TEMPLATE = """<?xml version="1.0" encoding="UTF-8"?>
<svg xmlns="http://www.w3.org/2000/svg"
     viewBox="0 0 1000 1000"
     width="1000" height="1000">
{paths}
</svg>"""


def extract_svg_paths(prediction: str) -> list[str]:
    """Извлекаем SVG path элементы из ответа модели."""
    paths = []

    # Полные <path> элементы
    path_elements = re.findall(r'<path\s[^>]*/>', prediction, re.DOTALL)
    if path_elements:
        return path_elements

    # Только данные path (строки, начинающиеся с M)
    for line in prediction.strip().split('\n'):
        line = line.strip()
        if line and re.match(r'^[MmLlHhVvCcSsQqTtAaZz]', line):
            paths.append(f'  <path d="{line}" fill="black"/>')

    return paths


# Сохраняем SVG файлы
saved = 0
failed = 0

for char, content in results.items():
    paths = extract_svg_paths(content)

    if not paths:
        print(f"  '{char}' — не удалось извлечь SVG пути")
        failed += 1
        continue

    svg_content = SVG_TEMPLATE.format(paths='\n'.join(paths))

    # Имя файла: u0410_А.svg
    safe_name = f"u{ord(char):04x}_{char}.svg"
    svg_path = SVG_DIR / safe_name
    svg_path.write_text(svg_content, encoding='utf-8')
    saved += 1

print(f"\nСохранено SVG: {saved}")
print(f"Не удалось извлечь: {failed}")
print(f"Директория: {SVG_DIR}")

## 6. Предпросмотр глифов

In [ ]:
from IPython.display import display, HTML, SVG as SVGDisplay
import base64

# Показываем сетку глифов
html = '<div style="display:flex; flex-wrap:wrap; gap:10px; background:#1a1a2e; padding:20px; border-radius:8px;">'

svg_files = sorted(SVG_DIR.glob('*.svg'))
for svg_file in svg_files:
    svg_content = svg_file.read_text(encoding='utf-8')
    # Извлекаем символ из имени файла
    match = re.match(r'u[0-9a-f]+_(.+)\.svg', svg_file.name)
    char = match.group(1) if match else '?'

    b64 = base64.b64encode(svg_content.encode()).decode()

    html += f'''
    <div style="text-align:center; width:90px;">
        <div style="color:#888; font-size:18px; margin-bottom:4px;">{char}</div>
        <div style="background:white; border-radius:4px; padding:5px; width:80px; height:80px; display:flex; align-items:center; justify-content:center;">
            <img src="data:image/svg+xml;base64,{b64}" style="max-width:70px; max-height:70px;"/>
        </div>
        <div style="color:#555; font-size:10px; margin-top:2px;">U+{ord(char):04X}</div>
    </div>'''

html += '</div>'

print(f"Предпросмотр {len(svg_files)} глифов:")
display(HTML(html))

## 7. Сборка шрифта (.ttf)

In [ ]:
from fontTools.fontBuilder import FontBuilder
from fontTools.pens.recordingPen import RecordingPen
from svgpathtools import parse_path
import xml.etree.ElementTree as ET

# Настройки шрифта
FONT_NAME = "KiriCallig"
FONT_FAMILY = "KiriCallig"
UPM = 1000
ASCENDER = 800
DESCENDER = -200
DEFAULT_WIDTH = 600
SVG_SIZE = 1000  # viewBox 0 0 1000 1000

FONT_DIR = OUTPUT_DIR / "font"
FONT_DIR.mkdir(parents=True, exist_ok=True)
FONT_PATH = FONT_DIR / f"{FONT_NAME}-{STYLE.capitalize()}.ttf"


def parse_svg_paths(svg_file):
    """Извлекаем d-атрибуты из SVG файла."""
    tree = ET.parse(svg_file)
    paths = []
    for elem in tree.iter():
        if elem.tag.endswith('path') or elem.tag == 'path':
            d = elem.get('d', '').strip()
            if d:
                paths.append(d)
    return paths


def draw_glyph(pen, svg_paths, scale=1.0):
    """Рисуем глиф из SVG path данных."""
    for d_string in svg_paths:
        path = parse_path(d_string)
        if not path:
            continue

        start = path[0].start
        pen.moveTo((
            round(start.real * scale),
            round((SVG_SIZE - start.imag) * scale)
        ))

        for seg in path:
            seg_type = type(seg).__name__
            if seg_type == 'Line':
                pen.lineTo((
                    round(seg.end.real * scale),
                    round((SVG_SIZE - seg.end.imag) * scale)
                ))
            elif seg_type == 'CubicBezier':
                pen.curveTo(
                    (round(seg.control1.real * scale), round((SVG_SIZE - seg.control1.imag) * scale)),
                    (round(seg.control2.real * scale), round((SVG_SIZE - seg.control2.imag) * scale)),
                    (round(seg.end.real * scale), round((SVG_SIZE - seg.end.imag) * scale)),
                )
            elif seg_type == 'QuadraticBezier':
                pen.qCurveTo(
                    (round(seg.control.real * scale), round((SVG_SIZE - seg.control.imag) * scale)),
                    (round(seg.end.real * scale), round((SVG_SIZE - seg.end.imag) * scale)),
                )
            elif seg_type == 'Arc':
                pen.lineTo((
                    round(seg.end.real * scale),
                    round((SVG_SIZE - seg.end.imag) * scale)
                ))

        if path.isclosed():
            pen.closePath()
        else:
            pen.endPath()


# Собираем карту глифов
glyph_map = {}
for svg_file in sorted(SVG_DIR.glob('*.svg')):
    match = re.match(r'u[0-9a-f]+_(.+)\.svg', svg_file.name, re.IGNORECASE)
    if match:
        char = match.group(1)
        if len(char) == 1:
            glyph_map[char] = svg_file

print(f"Найдено глифов: {len(glyph_map)}")

if not glyph_map:
    print("Нет SVG файлов для сборки!")
else:
    # Масштаб: SVG canvas -> font UPM
    scale = UPM / SVG_SIZE

    # Имена глифов и cmap
    glyph_names = ['.notdef', 'space']
    char_map = {32: 'space'}

    for char in sorted(glyph_map.keys()):
        gname = f'uni{ord(char):04X}'
        glyph_names.append(gname)
        char_map[ord(char)] = gname

    # Собираем шрифт
    fb = FontBuilder(UPM, isTTF=True)
    fb.setupGlyphOrder(glyph_names)
    fb.setupCharacterMap(char_map)
    fb.setupGlyf({})

    font = fb.font
    glyf_table = font['glyf']
    metrics = {'.notdef': (UPM, 0), 'space': (DEFAULT_WIDTH, 0)}

    print("\nСборка глифов:")
    for char, svg_file in sorted(glyph_map.items()):
        gname = f'uni{ord(char):04X}'
        svg_paths = parse_svg_paths(svg_file)

        if not svg_paths:
            print(f"  {char} — пусто, пропускаем")
            metrics[gname] = (DEFAULT_WIDTH, 0)
            continue

        rec_pen = RecordingPen()
        draw_glyph(rec_pen, svg_paths, scale)

        tt_pen = glyf_table.getPen(font.getGlyphSet(), gname)
        rec_pen.replay(tt_pen)
        metrics[gname] = (DEFAULT_WIDTH, 0)
        print(f"  {char} (U+{ord(char):04X}) — {len(svg_paths)} path(s)")

    # Метаданные
    fb.setupHorizontalMetrics(metrics)
    fb.setupHorizontalHeader(ascent=ASCENDER, descent=DESCENDER)
    fb.setupNameTable({
        'familyName': FONT_FAMILY,
        'styleName': STYLE.capitalize(),
        'psName': f'{FONT_NAME}-{STYLE.capitalize()}',
        'uniqueFontIdentifier': f'{FONT_NAME};1.0',
        'version': 'Version 1.0',
        'description': 'Cyrillic font inspired by Japanese calligraphy (Shodo)',
    })
    fb.setupOs2(
        sTypoAscender=ASCENDER,
        sTypoDescender=DESCENDER,
        sTypoLineGap=0,
        usWinAscent=ASCENDER,
        usWinDescent=abs(DESCENDER),
        sxHeight=500,
        sCapHeight=700,
        ulCodePageRange1=(1 << 2),  # Cyrillic
    )
    fb.setupPost()

    fb.font.save(str(FONT_PATH))
    print(f"\nШрифт сохранён: {FONT_PATH}")
    print(f"Размер: {FONT_PATH.stat().st_size / 1024:.1f} KB")

## 8. Тест шрифта и скачивание

In [ ]:
import base64
from IPython.display import HTML, display

if FONT_PATH.exists():
    # Встраиваем шрифт в HTML для предпросмотра
    font_b64 = base64.b64encode(FONT_PATH.read_bytes()).decode()

    sample_texts = [
        "Съешь ещё этих мягких французских булок, да выпей чаю",
        "АБВГДЕЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ",
        "абвгдежзийклмнопрстуфхцчшщъыьэюя",
        "0123456789",
        "Путь мастера начинается с одного штриха",
    ]

    samples_html = ''.join(
        f'<p style="font-size:{s}px; margin:10px 0;">{t}</p>'
        for t, s in zip(sample_texts, [36, 28, 28, 24, 32])
    )

    html = f"""
    <style>
        @font-face {{
            font-family: 'KiriCallig';
            src: url(data:font/truetype;base64,{font_b64}) format('truetype');
        }}
    </style>
    <div style="font-family: 'KiriCallig', serif; background: white;
                padding: 30px; border-radius: 8px; color: #1a1a1a;">
        <h2 style="text-align:center; color: #c0392b;">書 KiriCallig — {STYLE.capitalize()}</h2>
        {samples_html}
    </div>
    """
    display(HTML(html))
else:
    print("Шрифт не найден — проверь предыдущие шаги.")

In [ ]:
# Скачать шрифт и SVG-архив
from google.colab import files
import shutil

if FONT_PATH.exists():
    print("Скачивание шрифта...")
    files.download(str(FONT_PATH))

# Архив со всеми SVG
archive_path = OUTPUT_DIR / f"KiriCallig-{STYLE}-svgs"
shutil.make_archive(str(archive_path), 'zip', SVG_DIR)
print(f"\nАрхив SVG: {archive_path}.zip")
files.download(f"{archive_path}.zip")

## 9. Остановка сервера

In [ ]:
# Останавливаем vLLM сервер
if 'server_process' in dir() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=10)
    print("vLLM сервер остановлен.")
else:
    print("Сервер уже остановлен.")

---

## Что дальше?

- **Попробуй другие стили**: вернись к ячейке 0 и выбери `kaisho`, `gyosho`, `sosho` или `modern`
- **Ручная доработка**: открой SVG файлы в FontForge или Glyphs для правки
- **Несколько вариантов**: запусти генерацию с разной `TEMPERATURE` и выбери лучшие глифы
- **Расширение**: добавь украинские (ІЇЄҐіїєґ) или другие кириллические символы

### Заметки
- VecGlypher обучен на латинице (Google Fonts), кириллические символы — zero-shot
- Сложные буквы (Ж, Щ, Ы, Ъ) могут потребовать ручной доработки
- Для production-качества рекомендуется fine-tuning на кириллических данных